In [1]:
using Pkg
Pkg.activate("C:/Users/ibzja/Documents/UPF_2022_2026/4t/2n_trimestre/Practiques_tutelades/CellBasedModels.jl")
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
Makie.inline!(true)
using CSV, DataFrames, Statistics
using Printf, JLD2
using SpecialFunctions
using LsqFit
using LinearAlgebra

  Activating project at `C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl`


# DO SIMULATIONS

In [ ]:
fixed_source = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :fx => Float64,
        :fy => Float64,
        :W => Float64,
        :pressure => Float64,
        :active => Bool,

        :isSource => Bool, 
        :S => Float64,

        :methyl => Float64, #Receptor methylation
        :Yp => Float64, #CheYP levels, probability of tumblingç
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64
        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, #Energy parameters
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, #Cooperativity
        :Ki => Float64, #Dissociation constants
        :Ka => Float64,
        :τm => Float64, #Methylation adaptation timescale
        :α => Float64,      #Total Yp pool
        :ωFrec => Float64,     #Basal switching frequency
        :Ky => Float64,         #CheA - CheY phosphorylation rate
        :Z => Float64,          #CheZ concentration
        :Kz => Float64,         #CheZ mediated dephosphorylation rate
        :Yy => Float64,         #Basa Yp leak

        :DMedium => Float64,
        :delta => Float64,

        :sigma => Float64, #Gaussian width. 
        :R_source => Float64
    ),

    agentODE = quote  

        r = sqrt((x)^2 + (y)^2)
        ll = sqrt(DMedium / delta)

        # evitar singularidad en r=0
        r_eff = max(r, 1e-3)

        mm = 1.0 * besselk(0, r_eff / ll)

        M = mm

        F = ε0 + ε1 * methyl + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) #Equació del paper per definir activitat del receptor
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)        #Energia lliure en estat adaptat

        mx = (ε0 + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))      #CheYp segons activitat receptor

        dt(x) = vx 
        dt(y) = vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)        #Methylation
        
    end,

    agentRule = quote

        v_run = v
        v_tumble = 0.25 

        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        if active 
            λ = ωFrec*exp(-G)
            P = 1 - exp(-λ * dt)
                
        else
            λ = ωFrec*exp(G)
            P = 1 - exp(-λ * dt)                  
        end


        if active 
            λrt = ωFrec*exp(-G) 

            P_rt = 1 - exp(-λrt * dt)
            P = rand() 
                                                    #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
            if P < P_rt             #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
                active = false
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()           #Tumble = random reorientation
            else     #Si rate baixa 
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()       #Keep running, reorientation according to rotational difusion
            end

        else
            λtr = ωFrec*exp(G) 
            P_tr = 1 - exp(-λtr * dt)
            P = rand()

            if P < P_tr
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
            else
                active = false
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
            end
        end
    end,
    
    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler(),
    neighborsAlg=CBMNeighbors.CellLinked(cellEdge=10)
)

In [ ]:
ns = [0.25, 0.5, 0.75, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
steps = 60000
vel = [5.0, 10.0, 20.0, 100.0]


for n in ns
    for (idx, v) in enumerate(vel)
        println("Running n=$n")

        com = Community(
            fixed_source,
            N=1000,
            dt=0.01,
            simBox = [-100.0 100.0; -100.0 100.0],
            NMedium = [100, 100]
        )   

        m = 1/100
        g = 1/10000
        d = 1

        com.Dr_run = 0.062

        com.v = v 

        com.ωFrec = 1.3
        com.Ki = 0.0182
        com.Ka = 3.0
        com.Nrec = 6.0
        com.ε0   = 6.0
        com.ε1   = -1.0
        com.ε2   = 80
        com.ε3   = 80

        com.τm = 1  

        com.α   = 6.0

        com.K = 2.0 

        com.Ky = 100.0
        com.Kz = 10.0
        com.Z = 5.0
        com.Yy = 0.1

        com.m = 1.        
        com.d = 1.        
        com.l = 3;

        Dc = 10 
        delta = 0.01

        d = delta/((v/10)^2)

        #Canviar lambda

        com.DMedium = Dc / n
        com.delta = d * n

        com.x = 2.5*sqrt(com.DMedium/com.delta)
        com.y = 2.5*sqrt(com.DMedium/com.delta)
        com.theta = rand(Uniform(0,2π),com.N)

        com.methyl .= 0.0
        com.Yp .= com.K


        loadToPlatform!(com, preallocateAgents=1000)
        
        outfile = @sprintf("fsb_lambda_v%s_l%s.jld2", v, n)  ###CANVIARRR NOOM

        jldopen(outfile, "w") do file

            meta = JLD2.Group(file, "meta")
            meta["DMedium"] = com.DMedium
            meta["delta"] = com.delta

            for step in 1:steps
                step!(com)

                stepname = @sprintf("step_%06d", step)
                g = JLD2.Group(file, stepname)

                # Agent-level arrays (length = N)
                g["x"] = copy(com.x)
                g["y"] = copy(com.y)
                g["theta"] = copy(com.theta)

    
            end
        end
    end
end

# COMPARE VELOCITIES

In [ ]:
lambdas = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
steps = 60000

# Define the velocities to test
# velocities = [2.0]
velocities = [5.0, 10.0, 20.0, 100.0]
colors = [:red, :green, :blue, :black]

# Create a macrofigure with a grid layout
rows = 4
cols = 4
fig1 = Figure(size=(900 * cols, 600 * rows))

# Loop through each lambda value FIRST (creates the grid)
for (idx, lambda) in enumerate(lambdas)
    # Determine grid position
    row = ceil(Int, idx / cols)
    col = (idx - 1) % cols + 1
    
    # Create axis for this subplot
    ax = Axis(fig1[row, col], 
              xlabel="Time (s)", 
              ylabel="Relative Distance to Source", # Changed label to reflect normalization
              title="Λ = $lambda")
    
    # Loop through each velocity to plot on the SAME axis
    for (v_idx, v) in enumerate(velocities)
        color = colors[v_idx]
        try
            # Filename format: fsb_{v}_lambda_{lambda}.jld2
            filename = @sprintf("fsb_lambda_v%s_l%s.jld2", v, lambda)
        
            jldopen(filename, "r") do file 
                raw_dist_x = Float64[]
                # raw_dist_y = Float64[]

                meta_data = file["meta"]
                Dc = meta_data["DMedium"]
                delta = meta_data["delta"]
                l = sqrt(Dc/delta)
                
                for step in 1:steps
                    g = file[@sprintf("step_%06d", step)]

                    other_x = g["x"]
                    other_y = g["y"]

                    dx = other_x .- 0
                    dy = other_y .- 0
                    
                    dists_r = sqrt.(dx.^2 .+ dy.^2)
                    norm = dists_r ./ l

                    push!(raw_dist_x, mean(norm))
            
                end
                
                # Plot the normalized line
                lines!(ax, (1:steps)*0.01, raw_dist_x, color = color, label="$v μm/s")
                # lines!(ax, (1:steps)*0.01, raw_dist_y, color = color, label="$v μm/s")
    
            end
            if idx == 1 & v_idx == 4
                leg = Legend(fig1[:,5], ax, "Bacteria velocity")
            end

        catch e
            @warn "Could not load file for lambda $lambda, v=$v: $e"
        end
        
    end
end

display(fig1)

In [ ]:
save("Plots/Fixed_source_model/Redo/v_comparison_ABM.png", fig1)

# Separate populations

## Visualize individual tracks

In [ ]:
lambdas = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
vel = [5.0, 10.0, 20.0, 100.0]
n_files = length(lambdas)
steps = 60000

for (idx_v, v) in enumerate(vel)
    rows = 4
    cols = 4
    fig2 = Figure(size=(900 * cols, 600 * rows))

    for (idx, lambda) in enumerate(lambdas)
    
        filename = @sprintf("fsb_lambda_v%s_l%s.jld2", v, lambda)
        
        row = ceil(Int, idx / cols)
        col = (idx - 1) % cols + 1
        
        ax = Axis(fig2[row, col], 
                xlabel="Time (s)", 
                ylabel="Distance to source (X)",
                title="Λ = $lambda")

        jldopen(filename, "r") do file 
            for agent in 1:100
                mean_dist_x = Float64[] 
                for step in 1:10:steps
        
                    g = file[@sprintf("step_%06d", step)]

                    other_x = g["x"][agent]
                    dists_x = other_x .- 0

                    push!(mean_dist_x, mean(dists_x))
    
                end

                # Plotting X data
                lines!(ax, (1:10:steps)*0.01, mean_dist_x, label="Mean X")
            end
        end
    end
    save("Plots/Fixed_source_model/Redo/Trajectories_v$v_redo.png", fig2)
end

## Check valid agents

In [ ]:
function check_agent(file, step, agent_idx, threshold)

    data = file[@sprintf("step_%06d", step)]
    
    x = data["x"][agent_idx]
    y = data["y"][agent_idx]
    
    return (x <= threshold) && (y <= threshold)
end

In [ ]:
df_times = CSV.read("Files_threshold_time.csv", DataFrame)
df_times.lambda = replace.(df_times.lambda, "," => ".")

Row,v,eq_time,threshold,fit_start,fit_end,lambda
,Int64,Int64,Int64,Int64,Int64,String
1,10,3000,175,1000,2500,0.25
2,10,2000,75,250,1250,0.5
3,10,1000,75,200,600,0.75
4,10,1000,75,150,600,1
5,10,1000,75,100,390,2
6,10,1000,75,50,350,3
7,10,1000,75,50,225,4
8,10,1000,75,50,160,5
9,10,1000,75,50,110,6


In [ ]:
dfs_redo = Dict{Tuple{Float64, Float64}, DataFrame}()

lambdas = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
vel = [5.0, 10.0, 20.0, 100.0]
N_agents = 1000
end_step = 60000

for v in vel
    for lambda in lambdas

        filename = @sprintf("fsb_lambda_v%s_l%s.jld2", v, lambda)
        println("Processing $filename")

        # # lookup metadata
        row = df_times[df_times.filename .== filename, :]

        if nrow(row) == 0
            println("  ⚠️ Skipping (not found in df_times)")
            continue
        end

        start_step = row.eq_time[1]
        threshold_distance = row.threshold[1]

        -----------------------------
        STEP 1: find valid agents
        -----------------------------
        valid_agents = Int[]

        jldopen(filename, "r") do file
            for agent_idx in 1:N_agents
                is_valid = true

                for step in start_step:5:end_step
                    if !check_agent(file, step, agent_idx, threshold_distance)
                        is_valid = false
                        break
                    end
                end

                if is_valid
                    push!(valid_agents, agent_idx)
                end
            end
        end

        println("  ✔ valid agents: $(length(valid_agents))")

        # -----------------------------
        # STEP 2: extract trajectories
        # -----------------------------
        steps_vec = Int[]
        agents_vec = Int[]
        x_vec = Float64[]
        y_vec = Float64[]

        row = results[results.filename .== filename, :]
        valid_agents = row.valid_agents[1]

        jldopen(filename, "r") do file
            for agent in valid_agents
                for step in 1:10:end_step
                    g = file[@sprintf("step_%06d", step)]

                    push!(steps_vec, step)
                    push!(agents_vec, agent)
                    push!(x_vec, g["x"][agent])
                    push!(y_vec, g["y"][agent])
                end
            end
        end

        df = DataFrame(
            step = steps_vec,
            agent = agents_vec,
            x = x_vec,
            y = y_vec
        )

        # store in dict
        dfs_redo[(v, lambda)] = df

    end
end